In [14]:
import pandas as pd
df = pd.read_csv("dirty_cafe_sales.csv")
print(df.shape)
print(df.head(10))
print(df.info())
print(df.isnull().sum())
print(df.duplicated().sum())

for col in ["Item" , "Payment Method","Location"]:
    print(f"---{col}---")
    print(df[col].value_counts(dropna=False))
    print()


for col in df.columns:
  n = df[col].isin(["ERROR" , "UNKNOWN"]).sum()
  print(col, "->" , n)

print("Duplicate IDs:",df["Transaction ID"].duplicated().sum())


clean = df.copy()   # always keep the raw data untouched

clean = clean.replace(["ERROR", "UNKNOWN"], pd.NA)

for col in ["Transaction ID", "Item", "Payment Method", "Location"]:
    clean[col] = clean[col].str.strip()
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

clean["Transaction Date"] = pd.to_datetime(clean["Transaction Date"], errors="coerce")

print(clean.dtypes)

# 1. Fill missing price using the item's usual price
price_map = clean.groupby("Item")["Price Per Unit"].agg(lambda s: s.mode().iloc[0])
clean["Price Per Unit"] = clean["Price Per Unit"].fillna(clean["Item"].map(price_map))

# 2. Total = Quantity x Price
m = clean["Total Spent"].isna()
clean.loc[m, "Total Spent"] = clean["Quantity"] * clean["Price Per Unit"]

# 3. Quantity = Total / Price
m = clean["Quantity"].isna()
clean.loc[m, "Quantity"] = clean["Total Spent"] / clean["Price Per Unit"]

# 4. Price = Total / Quantity
m = clean["Price Per Unit"].isna()
clean.loc[m, "Price Per Unit"] = clean["Total Spent"] / clean["Quantity"]

for col in ["Item", "Payment Method", "Location"]:
    clean[col] = clean[col].fillna("Unknown")

print("Shape before:", df.shape, "| after:", clean.shape)
print(clean.isnull().sum())
print(clean.describe())
print("Duplicate IDs:", clean["Transaction ID"].duplicated().sum())

known = clean[clean["Item"] != "Unknown"]
n_items = known.groupby("Price Per Unit")["Item"].nunique()
print(n_items)   # how many different items share each price

unique_prices = n_items[n_items == 1].index
price_to_item = (known[known["Price Per Unit"].isin(unique_prices)]
                 .groupby("Price Per Unit")["Item"].first())
print(price_to_item)

mask = clean["Item"] == "Unknown"
clean.loc[mask, "Item"] = clean.loc[mask, "Price Per Unit"].map(price_to_item).fillna("Unknown")
print("Item still Unknown:", (clean["Item"] == "Unknown").sum())

calc = clean["Quantity"] * clean["Price Per Unit"]
print("Total mismatches:", (calc.notna() & (calc != clean["Total Spent"])).sum())
print("Fractional quantities:", (clean["Quantity"].dropna() % 1 != 0).sum())

before = len(clean)
clean = clean.dropna(subset=["Total Spent"])
print("Rows dropped (no Total Spent):", before - len(clean))

print(clean.isnull().sum())

clean["Quantity"] = clean["Quantity"].astype("Int64")
clean.to_csv("cleaned_data.csv", index=False)

from google.colab import files
files.download("cleaned_data.csv")

raw = df.copy()
raw_clean = raw.replace(["ERROR", "UNKNOWN"], pd.NA)

summary = pd.DataFrame({
    "Missing before": raw_clean.isnull().sum(),
    "Missing after":  clean.replace("Unknown", pd.NA).isnull().sum(),
})
print(summary)
print("Rows before:", len(raw), "| Rows after:", len(clean))
print("Rows dropped:", len(raw) - len(clean))












(10000, 8)
  Transaction ID      Item Quantity Price Per Unit Total Spent  \
0    TXN_1961373    Coffee        2            2.0         4.0   
1    TXN_4977031      Cake        4            3.0        12.0   
2    TXN_4271903    Cookie        4            1.0       ERROR   
3    TXN_7034554     Salad        2            5.0        10.0   
4    TXN_3160411    Coffee        2            2.0         4.0   
5    TXN_2602893  Smoothie        5            4.0        20.0   
6    TXN_4433211   UNKNOWN        3            3.0         9.0   
7    TXN_6699534  Sandwich        4            4.0        16.0   
8    TXN_4717867       NaN        5            3.0        15.0   
9    TXN_2064365  Sandwich        5            4.0        20.0   

   Payment Method  Location Transaction Date  
0     Credit Card  Takeaway       2023-09-08  
1            Cash  In-store       2023-05-16  
2     Credit Card  In-store       2023-07-19  
3         UNKNOWN   UNKNOWN       2023-04-27  
4  Digital Wallet  In-store

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

                  Missing before  Missing after
Transaction ID                 0              0
Item                         969            477
Quantity                     479              3
Price Per Unit               533              3
Total Spent                  502              0
Payment Method              3178           3169
Location                    3961           3953
Transaction Date             460            460
Rows before: 10000 | Rows after: 9977
Rows dropped: 23


In [15]:
raw = df.copy()
raw_clean = raw.replace(["ERROR", "UNKNOWN"], pd.NA)

summary = pd.DataFrame({
    "Missing before": raw_clean.isnull().sum(),
    "Missing after":  clean.replace("Unknown", pd.NA).isnull().sum(),
})
print(summary)
print("Rows before:", len(raw), "| Rows after:", len(clean))
print("Rows dropped:", len(raw) - len(clean))

                  Missing before  Missing after
Transaction ID                 0              0
Item                         969            477
Quantity                     479              3
Price Per Unit               533              3
Total Spent                  502              0
Payment Method              3178           3169
Location                    3961           3953
Transaction Date             460            460
Rows before: 10000 | Rows after: 9977
Rows dropped: 23


In [16]:
from google.colab import files
df.to_csv("raw_data.csv", index=False)
files.download("raw_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>